# Project Attribution and Acknowledgments


## Author Information
* **Researcher:** Aman Satyendra Yadav
* **Program:** M.Sc. Life Science Informatics (LSI)
* **Academic Home:** Faculty of Computer Science, [TH Deggendorf (DIT)](https://www.th-deg.de/lsi-m).


## Institutional Affiliation &  Supervision
This research was conducted as a collaborative Master’s project between the Faculty of Computer Science at the Technische Hochschule Deggendorf (DIT) and the Biomedical Center (BMC) at LMU Munich.

The project was hosted by the Bioinformatics Core Facility within the Division of Molecular Biology at the BMC. The BMC provided the high-performance computing (HPC) infrastructure and the experimental data support essential for this work. Dr. Tobias Straub (Head of Bioinformatics Core, BMC) provided the conceptual framework for the study and offered structural and methodological guidance throughout the development of the predictive workflow.

Prof. Dr. Melanie Kappelmann-Fenzl (Genomics and Biomedical Data Science, DIT) served as the internal academic supervisor, providing expert guidance on the formal development of the Master’s thesis and ensuring the research aligns with the academic standards of the Life Science Informatics program.

While the conceptual and academic framework was established by the supervisors, the author (Aman Satyendra Yadav) is solely responsible for the technical implementation, software logic, and data processing pipelines developed in this project. Accordingly, the author should be the primary point of contact for any inquiries or technical questions regarding the software architecture and implementation details. The success of this project is a result of the synergy between the author's technical execution, Dr. Straub’s structural support, and Prof. Dr. Kappelmann-Fenzl’s academic oversight."

## Abstract

### **Biological Scope and Objective**


This research focuses on the computational prediction of Transcription Factor (TF) binding sites within the Drosophila melanogaster model organism (dm6). Transcription factors play a critical role in the spatial and temporal regulation of gene expression; however, mapping their global binding landscape via experimental methods like ChIP-seq is resource-intensive. This project aims to overcome these limitations by leveraging a multi-modal deep learning architecture designed to integrate various multi-omic features, thereby achieving high-resolution and highly accurate site-specific predictions.

### Phase I: Multi-omic Data Preprocessing & Pipeline Engineering
The first phase of the project involves the extensive processing of raw multi-omic datasets, including ChIP-seq (ground truth), ATAC-seq (chromatin accessibility), and Whole Genome Sequencing (WGS) for sequence extraction.**

Computational Environment: These data-intensive tasks were executed on the High-Performance Computing (HPC) infrastructure at the LMU Biomedical Center (BMC).

Methodological Framework: The preprocessing workflow was modeled after the protocols established in the core reference article: “Multimodal learning decodes the global binding landscape of chromatin-associated proteins.” This phase ensured the generation of high-quality genomic tensors, standardized for input into neural network architectures.

### Phase II: Comparative Model Development and Evaluation
The second phase of the research focuses on the continuation of the pipeline into three distinct deep learning models.

The first model focuses on the ATAC feature and its role in predicting the quantitative ChIP signals in the held-out chromosome of Chr2R. For extra context, TN5 fragments were first classified into three groups according to their length in base pairs. The groups are short fragments, mononucleosomes, and dinucleosomal fragments. These channels provide extra context to the ATAC model and help to achieve the maximum contextual accuracy possible.

The second model is a DNA-only model which only uses the one-hot encoded DNA. It has 4 channels for all four DNA nucleotides.

The end model focuses on combining both the ATAC with TN5 fragments as well as the DNA model. This model provides excellent accuracy and can be used to compare baseline performance between all three models.

### Crosschecking Model Performance

In the end, in situ permutation is done to crosscheck the pipeline. This step checks if the model is really learning biological concepts or if it is just memorizing noise.

# Modules and config file

This phase coordinates the import of all discrete scripts required to execute the multimodal deep learning workflow, where each script addresses a distinct analytical objective. These computational modules operate independently of one another, yet they maintain a strict dual dependency on both the GenomicLoader instance and the central configuration file. The configuration architecture encapsulates all global hyperparameters, execution parameters, and absolute system paths required to seamlessly interface the preprocessed data structures with the downstream machine learning pipeline.

In [ ]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CUDNN_USE_AUTOTUNE"] = "0"

import tensorflow as tf

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("GPUs:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

tf.config.set_soft_device_placement(True)

In [ ]:
# extra impotrs (for safety)
import random
import yaml
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from DataLoader3 import GenomicDataLoader
from Visualizer3 import Visualizer
from ATAC_model3 import ATACPredictionModel
from DNA_model3 import DNAPredictionModel
from ATAC_DNA_model3 import MultimodalGenomicModel
from Insito_mutation3 import GenomicPermuter

%matplotlib inline


with open("config_file_ml.yaml", "r") as file:
    data = yaml.safe_load(file)


def set_global_seed(seed):
    seed = int(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

    print(f"Global random seed: {seed}")


MODEL_TYPE = "atac"   # dna, atac or multimodal
SEED_INDEX = 0

seeds = data["params"]["model_seeds"][MODEL_TYPE]

if not seeds:
    raise ValueError(f"No seeds configured for {MODEL_TYPE}.")
if not 0 <= SEED_INDEX < len(seeds):
    raise ValueError("SEED_INDEX is outside the configured seed list.")

seed = int(seeds[SEED_INDEX])
set_global_seed(seed)

print(f"Selected model: {MODEL_TYPE}")
print(f"Selected seed: {seed}")
print("Configuration and dependencies successfully imported.")

# Genomic Loader
The Genomic Data Loader connects the preprocessing workflow to the downstream analysis and machine-learning scripts. It reads the processed genomic files from the paths specified in the configuration file and loads the required ATAC-seq, ChIP-seq, DNA-sequence, chromosome, and track-name information into memory.

The loader aligns the different data types using the same genomic-window order. This alignment is essential because the DNA sequence, chromatin features, and ChIP-seq targets for each array index must represent the same genomic region. It also combines the signal tracks into structured arrays and provides chromosome labels that are later used to create separate training, validation, and test sets.

The loader itself does not train the models or perform prediction. Instead, it prepares and stores the input arrays required by the visualization, DNA-only, ATAC-based, multimodal, and in-silico mutation modules. Therefore, the relevant loader methods must be executed before these downstream classes are initialized. If an input file is missing, the channel dimensions are inconsistent, or the genomic windows are not aligned, the subsequent analysis cannot be performed reliably.

In [ ]:
# 1. Initialize (Loads config automatically)
loader = GenomicDataLoader("config_file_ml.yaml")

loader.read_one_hot_encode()

loader.read_chip_data()

loader.read_atac_data()

loader.add_chrmos() 

loader.get_shape()

loader.chip_concantor()

full_fragments = loader.read_all_atac_fragments_3d()







loader.verify_data_normalization() # this loads the data this one now fix it 



# Diagnostics and Visual Exploration Module
This optional visualization module displays the relationships among ATAC-seq and ChIP-seq signals at window, chromosome, and whole-genome levels. The plots support data interpretation and quality assessment but do not contribute directly to model training or prediction.

The visualizations show how chromatin accessibility and protein-binding signals vary across genomic regions. Some accessible regions contain strong ChIP-seq signals, whereas others show weak or no binding. Similarly, ChIP-seq enrichment may occur in regions with comparatively low ATAC-seq signal. These patterns indicate statistical associations rather than direct causal effects. Protein specificity, cofactors, regulatory context, and technical variation may also influence the observed signals.

In [ ]:
visualizer = Visualizer.from_loader(loader)

total_windows = visualizer.concatenated_data.shape[0]
total_chips = visualizer.concatenated_data.shape[-1] - 1
window = min(7344, total_windows - 1)


# plot ATAC and four ChIP tracks
visualizer.ATAC_vs_CHIP(
    window_idx=window,
    limit=min(5, total_chips + 1)
)


# find and plot active windows
active, _ = visualizer.active_windows(
    threshold_atac=0.2,
    threshold_chip=0.2
)

if len(active) > 0:
    visualizer.ATAC_vs_CHIP_multiple_windows(
        window_list=active[:5],
        limit=False
    )
else:
    print("No active windows found.")


# chromosome distribution
visualizer.plot_chromosomal_distribution(
    threshold_atac=0.1,
    threshold_chip=0.1
)


# ATAC and ChIP correlation
results = visualizer.atac_chip_window_correlation(
    limit=True,
    summary_stat="max",
    method="pearson",
    save_csv=False,
    save_plot=False
)

print(results)


# whole-genome view
visualizer.whole_genome_view(
    chip_tracks=range(total_chips),
    summary_stat="mean",
    max_points=1000
)

# Model I: Sequence-Only Architecture (DNA-Only Baseline)
This model utilizes a one-hot encoded DNA sequence dataset to predict continuous, quantitative ChIP-seq signal intensities across the Drosophila melanogaster (dm6) reference genome. It stands as one of the primary foundational architectures within the overall computational pipeline.



In [ ]:
import gc
import numpy as np
import tensorflow as tf

seeds = data["params"]["model_seeds"]["dna"]
batch_size = data["params"]["batch_size_dna"]
all_results = []

for seed in seeds:
    print(f"\nDNA model — seed {seed}")

    tf.keras.backend.clear_session()
    set_global_seed(seed)

    dna = DNAPredictionModel.from_loader(loader)
    dna.Train_test_Split("config_file_ml.yaml")

    model, history = dna.DNA_model_sequential(
        config_path="config_file_ml.yaml",
        seed=seed
    )

    test = model.evaluate(
        dna.X_dna_test, dna.Y_dna_test,
        batch_size=batch_size,
        return_dict=True
    )

    predictions = model.predict(
        dna.X_dna_test,
        batch_size=batch_size
    )

    pearson = dna.pearson_cor_summary(predictions)
    wasserstein = dna.DNA_cal_wasserstein(predictions)
    jsd = dna.DNA_cal_jsd(predictions)

    results = {
        "seed": seed,
        "loss": test["loss"],
        "mae": test["mae"],
        "pooled_pearson": pearson["pooled_pearson"],
        "mean_pearson": pearson["mean_pearson"],
        "wasserstein_bp": wasserstein["avg_wasserstein"],
        "jsd": jsd["avg_jsd"],
        "epochs": len(history.history["loss"])
    }

    all_results.append(results)

    for name, value in results.items():
        print(f"{name}: {value}")

    del dna, model, history, predictions
    gc.collect()


print("\nMean ± SD across seeds")

for metric in all_results[0]:
    if metric == "seed":
        continue

    values = np.array([result[metric] for result in all_results])
    print(
        f"{metric}: {values.mean():.6f} ± "
        f"{values.std(ddof=1):.6f}"
    )

# Model II: Chromatin Accessibility Architecture (ATAC-Only Baseline + TN5 fragments ) 

This model predicts continuous ChIP-seq profiles using chromatin-derived information without DNA-sequence input. It combines the ATAC-seq accessibility signal with Tn5 fragment-length channels representing nucleosome-free, mononucleosomal, and dinucleosomal regions. The model therefore evaluates how effectively local chromatin accessibility and organization alone can predict transcription-factor binding across the Drosophila melanogaster dm6 genome. Its performance provides a chromatin-based baseline for comparison with the DNA-only and multimodal models.

In [ ]:
import importlib
import ATAC_model3

importlib.reload(ATAC_model3)


atac_results, atac_summary = ATAC_model3.run_atac_seed_experiments(
    loader=loader,
    TN5_frags=full_fragments,
    config_path="config_file_ml.yaml"
)


print("\nResults for each seed")

for result in atac_results:
    print(f"\nSeed {result['seed']}")

    for metric, value in result.items():
        if metric != "model":
            print(f"{metric}: {value}")


print("\nOverall ATAC + Tn5 results")

for metric, values in atac_summary.items():
    print(f"{metric}: {values['mean']:.6f} ± {values['sd']:.6f}")

# multimodal acr 
Thi model combines both DNA and ATAC to predict chip signal across the Genome , This fall under the last step of ytaining Ml pipe line 


In [ ]:
import importlib
import ATAC_DNA_model3

importlib.reload(ATAC_DNA_model3)


multimodal_results, multimodal_summary = (
    ATAC_DNA_model3.multimodal_seed_execution(
        loader=loader,
        TN5_frags=full_fragments,
        config_path="config_file_ml.yaml"
    )
)


print("\nResults for each seed")

for result in multimodal_results:
    print(f"\nSeed {result['seed']}")

    for metric, value in result.items():
        print(f"{metric}: {value}")


print("\nOverall multimodal results")

for metric, values in multimodal_summary.items():
    print(f"{metric}: {values['mean']:.6f} ± {values['std']:.6f}")

# In-silico analysis
The in-silico analysis examines whether the multimodal model responds meaningfully to changes in its DNA and chromatin inputs. The correct-versus-shuffled plots compare model performance before and after disrupting the pairing between genomic inputs and ChIP-seq targets. The permutation histogram further compares the observed Pearson correlation with a null distribution generated from repeated target shuffling.

The DNA mutation heatmap shows how predicted ChIP-seq signals change after substituting individual bases at each sequence position. The chromatin perturbation curves show the model response when ATAC and Tn5 signals are scaled within regions of different sizes. These plots measure model sensitivity and input dependence; they should not be interpreted as direct evidence of causal biological effects. Further interpretation is provided in the in-silico perturbation and limitations sections of the thesis report.

In [ ]:
import importlib
import numpy as np
import tensorflow as tf
import Insito_mutation3

importlib.reload(Insito_mutation3)
GenomicPermuter = Insito_mutation3.GenomicPermuter

seed = 11
batch_size = data["params"]["batch_size_multimodal"]


# prepare test data
multimodal = MultimodalGenomicModel.from_loader(loader)

multimodal.Train_Test_Split(
    config_path="config_file_ml.yaml",
    TN5_frags=full_fragments
)


# recreate weighted loss
settings = data["params"]["multimodal_loss_tuning"]
threshold = settings["multimodal_peak_threshold"]
peak_weight = settings["multimodal_peak_weight_multiplier"]
background_weight = settings["multimodal_background_penalty"]


def weighted_mse(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    weights = tf.where(
        y_true >= threshold,
        peak_weight,
        1.0
    )

    background = tf.where(
        (y_true == 0.0) & (y_pred > 0.01),
        background_weight,
        1.0
    )

    return tf.reduce_mean(
        tf.square(y_true - y_pred)
        * weights
        * background
    )


# load saved multimodal model
model_path = (
    f"checkpoints/multimodal/seed_{seed}/"
    "best_atac_dna_model.keras"
)

multimodal.model = tf.keras.models.load_model(
    model_path,
    compile=False
)

multimodal.model.compile(
    optimizer="adam",
    loss=weighted_mse,
    metrics=["mae"]
)

print(f"Loaded model: {model_path}")


# initialize permutation analysis
permuter = GenomicPermuter(seed=seed)

test_inputs = [
    multimodal.X_Atac_test,
    multimodal.X_dna_test
]

test_targets = multimodal.Y_multi_test

permuter.fit_transformer_random(
    test_inputs,
    test_targets
)


# correct and shuffled target comparison
true_results, shuffled_results = permuter.evaluate_models(
    trained_model=multimodal.model,
    batch_size=batch_size,
    max_correlation_windows=1000
)

print("\nCorrect pairing:", true_results)
print("Shuffled pairing:", shuffled_results)


# correct-versus-shuffled graphs
permuter.plot_comparison(
    target_metric="mae",
    save_path="shuffled_mae.png"
)

permuter.plot_comparison(
    target_metric="pearson_r",
    save_path="shuffled_pearson.png"
)


# repeated permutation test
permutation_results = permuter.permutation_correlation_test(
    trained_model=multimodal.model,
    n_permutations=100,
    batch_size=batch_size,
    max_windows=1000
)

print("\nPermutation test")
print(
    "Observed Pearson:",
    permutation_results["observed_mean_track_pearson"]
)
print("Null mean:", permutation_results["null_mean"])
print("Null SD:", permutation_results["null_sd"])
print("P-value:", permutation_results["empirical_p_value"])


# DNA mutation test and heatmap
dna_impact = permuter.DNA_mutation_test(
    trained_model=multimodal.model,
    start_idx=0,
    end_idx=2,
    plot=True
)

print("\nDNA mutation")
print("Mean change:", np.mean(dna_impact))
print("Maximum change:", np.max(dna_impact))


# select the strongest chromatin window
signal = np.mean(
    np.abs(multimodal.X_Atac_test),
    axis=(1, 2)
)

window = int(np.argmax(signal))


# ATAC and Tn5 perturbation
chromatin_results = permuter.ATAC_mutation_test(
    trained_model=multimodal.model,
    multipliers=[0.0, 0.5, 1.0, 1.5, 2.0],
    radii=[50, 250, 500],
    target_idx=window
)

print(f"\nChromatin perturbation window: {window}")

for radius, predictions in chromatin_results.items():
    original = predictions[1.0]

    for multiplier, prediction in predictions.items():
        change = np.mean(
            np.abs(prediction - original)
        )

        print(
            f"Radius {radius}, multiplier {multiplier}: "
            f"{change:.6f}"
        )


# ATAC and Tn5 perturbation graph
permuter.plot_dimmer_curves(
    trained_model=multimodal.model,
    curve_data=chromatin_results,
    target_idx=window,
    save_path="chromatin_perturbation.png"
)

In [ ]:
!grep -n "^class " /work/project/becstr_012/Ms_project/Testing/ResidualATACFusionModel.py